# run_future_pipeline_6basins notebook

This notebook embeds the full code from `run_future_pipeline_6basins.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Run bias correction, future DPWM, and deficit plots for the six example basins.

  python run_future_pipeline_6basins.py --extraction areal --workers 2
  python run_future_pipeline_6basins.py --extraction nearest --workers 2
  python compare_extraction_methods.py
"""

from __future__ import annotations

import argparse
import subprocess
import sys
from pathlib import Path

from calibration_io import BASIN_IDS
from compare_extraction_methods import dirs_for_method

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CORDEX_BASE = Path(r"F:\Data\EUROCORDEX_MPI")
BASINS_CSV = ",".join(BASIN_IDS)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument(
        "--extraction",
        choices=("areal", "nearest"),
        default="areal",
        help="Spatial extraction for bias correction (default: areal).",
    )
    p.add_argument("--workers", type=int, default=2, help="Parallel bias-correction workers (default 2).")
    p.add_argument("--cordex_base", type=Path, default=CORDEX_BASE)
    p.add_argument("--skip_bias", action="store_true")
    p.add_argument("--skip_dpwm", action="store_true")
    p.add_argument("--skip_deficits", action="store_true")
    p.add_argument(
        "--skip_existing_bias",
        action="store_true",
        help="Skip basins that already have pr/tas/pet_future_bc CSVs.",
    )
    return p.parse_known_args()[0]


def run_step(label: str, cmd: list[str]) -> None:
    print(f"\n{'=' * 60}\n{label}\n{'=' * 60}", flush=True)
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=ROOT, check=True)


def main() -> None:
    args = parse_args()
    py = sys.executable
    paths = dirs_for_method(args.extraction, ROOT)
    bc_dir = paths["bc"]
    discharge_dir = paths["discharge"]
    deficit_dir = paths["deficit"]

    if not args.skip_bias:
        cmd = [
            py,
            str(ROOT / "bias_correction.py"),
            "--basins",
            BASINS_CSV,
            "--extraction",
            args.extraction,
            "--workers",
            str(args.workers),
            "--cordex_base",
            str(args.cordex_base),
            "--out_dir",
            str(bc_dir),
        ]
        if args.skip_existing_bias:
            cmd.append("--skip_existing")
        run_step(f"Step 1/3: Bias correction ({args.extraction}, parallel)", cmd)

    if not args.skip_dpwm:
        run_step(
            "Step 2/3: Future forcing + DPWM",
            [
                py,
                str(ROOT / "run_future_dpwm.py"),
                "--root",
                str(ROOT),
                "--basins",
                BASINS_CSV,
                "--bc_dir",
                str(bc_dir),
                "--discharge_out",
                str(discharge_dir),
                "--forcing_out",
                str(paths["forcing"]),
            ],
        )

    if not args.skip_deficits:
        run_step(
            "Step 3/3: Water deficits + plots",
            [
                py,
                str(ROOT / "compute_q80_deficits.py"),
                "--root",
                str(ROOT),
                "--basins",
                BASINS_CSV,
                "--plot",
                "--q_csv",
                str(discharge_dir / "Q_total_all_basins.csv"),
                "--out_dir",
                str(deficit_dir),
            ],
        )

    print("\nAll steps finished.")
    print(f"  Extraction: {args.extraction}")
    print(f"  Bias-corrected forcing: {bc_dir}")
    print(f"  Future Q: {discharge_dir / 'Q_total_all_basins.csv'}")
    print(f"  Deficit plots: {deficit_dir / 'plots'}")


# Execute the script entry point
main()
